In [ ]:
# ==========================================
# PROYECTO: EXTRACCIÓN DE DATOS URBANOS
# COMUNA: San Miguel, Santiago, Chile
# Autor: Enfoque Data Science + Ingeniería Urbana
# ==========================================

# 1. Instalación (ejecutar una vez)
# !pip install osmnx geopandas shapely fiona pyproj

In [ ]:
# 2. Librerías
import osmnx as ox
import geopandas as gpd
import os

In [ ]:
# 3. Configuración
ox.settings.log_console = True
ox.settings.use_cache = True

# Carpeta de salida
output_dir = "san_miguel_capas"
os.makedirs(output_dir, exist_ok=True)

# Área de estudio
place_name = "San Miguel, Santiago, Chile"

# Sistema de coordenadas proyectado (Chile)
CRS_PROY = "EPSG:32719"

print("Descargando datos para:", place_name)

In [ ]:
# ==========================================
# 4. RED VIAL (GRAFO)
# ==========================================
G = ox.graph_from_place(place_name, network_type="drive")
nodes, edges = ox.graph_to_gdfs(G)

edges = edges.to_crs(CRS_PROY)
nodes = nodes.to_crs(CRS_PROY)

edges.to_file(f"{output_dir}/red_vial.shp")
nodes.to_file(f"{output_dir}/nodos_viales.shp")

print("✔ Red vial descargada")

In [ ]:
# ==========================================
# 5. NODOS CRÍTICOS (INTERSECCIONES)
# ==========================================
nodos_criticos = nodes[nodes["street_count"] >= 4]
nodos_criticos.to_file(f"{output_dir}/nodos_criticos.shp")

print("✔ Nodos críticos identificados")

In [ ]:
# ==========================================
# 6. EQUIPAMIENTO URBANO (POIs)
# ==========================================
tags_pois = {
    "amenity": True,
    "shop": True,
    "tourism": True
}

pois = ox.features_from_place(place_name, tags_pois)

if not pois.empty:
    pois = pois.to_crs(CRS_PROY)
    pois.to_file(f"{output_dir}/equipamiento_urbano.shp")
    print("✔ POIs descargados")
else:
    print("⚠ No se encontraron POIs")

In [ ]:
# ==========================================
# 7. PARADEROS DE BUS
# ==========================================
tags_bus = {"highway": "bus_stop"}
bus_stops = ox.features_from_place(place_name, tags_bus)

if not bus_stops.empty:
    bus_stops = bus_stops.to_crs(CRS_PROY)
    bus_stops.to_file(f"{output_dir}/paraderos.shp")
    print("✔ Paraderos descargados")

In [ ]:
# ==========================================
# 8. INFRAESTRUCTURA PEATONAL
# ==========================================
tags_walk = {"highway": ["footway", "pedestrian", "path"]}
walkways = ox.features_from_place(place_name, tags_walk)

if not walkways.empty:
    walkways = walkways.to_crs(CRS_PROY)
    walkways.to_file(f"{output_dir}/infraestructura_peatonal.shp")
    print("✔ Infraestructura peatonal descargada")

In [ ]:
# ==========================================
# 9. CICLOVÍAS
# ==========================================
tags_cycle = {"cycleway": True}
cycleways = ox.features_from_place(place_name, tags_cycle)

if not cycleways.empty:
    cycleways = cycleways.to_crs(CRS_PROY)
    cycleways.to_file(f"{output_dir}/ciclovias.shp")
    print("✔ Ciclovías descargadas")

In [ ]:
# ==========================================
# 10. USO DE SUELO
# ==========================================
tags_landuse = {"landuse": True}
landuse = ox.features_from_place(place_name, tags_landuse)

if not landuse.empty:
    landuse = landuse.to_crs(CRS_PROY)
    landuse.to_file(f"{output_dir}/uso_suelo.shp")
    print("✔ Uso de suelo descargado")

In [ ]:
# ==========================================
# 11. LIMPIEZA BÁSICA (OPCIONAL)
# ==========================================
# Eliminar geometrías inválidas
def limpiar_gdf(gdf):
    return gdf[gdf.geometry.notnull()]

In [ ]:
# ==========================================
# 12. RESUMEN FINAL
# ==========================================
print("\\n===== RESUMEN =====")
print(f"Nodos totales: {len(nodes)}")
print(f"Calles totales: {len(edges)}")
print(f"Nodos críticos: {len(nodos_criticos)}")

print("\\nArchivos guardados en carpeta:", output_dir)
print("Proceso completado correctamente")